# 01 - Extract OSM Infrastructure

## Objective

This notebook extracts only the required infrastructure from the six
OpenStreetMap regional PBF files.

Infrastructure extracted:

- Hospitals
- Clinics
- Schools
- Fire Stations

The extracted infrastructure is merged into a single GeoDataFrame and
saved as a GeoParquet file for fast spatial querying in later notebooks.

Output:

datasets/
    india_infrastructure.parquet

In [1]:
from pathlib import Path
from tqdm.auto import tqdm

import gc
import warnings

import pandas as pd
import geopandas as gpd

from pyrosm import OSM

warnings.filterwarnings("ignore")

In [2]:
PROJECT_DIR = Path.cwd().parent

DATASET_DIR = PROJECT_DIR / "datasets"

OSM_DIR = DATASET_DIR / "osm_regions"

OUTPUT_FILE = (
    DATASET_DIR /
    "india_infrastructure.parquet"
)

CHECKPOINT_DIR = (
    DATASET_DIR /
    "osm_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    exist_ok=True
)

In [3]:
REGION_FILES = sorted(

    OSM_DIR.glob("*.osm.pbf")

)

print(

    "Total Regional Files:",

    len(REGION_FILES)

)

for file in REGION_FILES:

    print(file.name)

Total Regional Files: 6
central-zone-260710.osm.pbf
eastern-zone-260710.osm.pbf
north-eastern-zone-260710.osm.pbf
northern-zone-260710.osm.pbf
southern-zone-260710.osm.pbf
western-zone-260710.osm.pbf


In [4]:
INFRASTRUCTURE_FILTER = {

    "amenity":[

        "hospital",

        "clinic",

        "school",

        "fire_station"

    ]

}

In [5]:
def extract_region(region_file):

    print()

    print("="*60)

    print(region_file.name)

    print("="*60)

    osm = OSM(

        str(region_file)

    )

    pois = osm.get_pois(

        custom_filter=INFRASTRUCTURE_FILTER

    )

    if pois is None:

        print(

            "Nothing extracted."

        )

        return gpd.GeoDataFrame()

    pois = pois.to_crs(

        "EPSG:4326"

    )

    pois["RegionFile"] = region_file.name

    return pois

In [6]:
all_regions = []

for region in tqdm(

    REGION_FILES

):

    try:

        gdf = extract_region(

            region

        )

        if len(gdf):

            all_regions.append(

                gdf

            )

        del gdf

        gc.collect()

    except Exception as e:

        print(

            e

        )

  0%|          | 0/6 [00:00<?, ?it/s]


central-zone-260710.osm.pbf
Unable to allocate 444. MiB for an array with shape (58163316,) and data type object

eastern-zone-260710.osm.pbf

north-eastern-zone-260710.osm.pbf

northern-zone-260710.osm.pbf

southern-zone-260710.osm.pbf


western-zone-260710.osm.pbf


In [7]:
print("Regions loaded:", len(all_regions))

for i, gdf in enumerate(all_regions):
    print(f"Region {i+1}: {len(gdf):,} features")

Regions loaded: 4
Region 1: 7,814 features
Region 2: 2,349 features
Region 3: 16,921 features
Region 4: 20,477 features


In [8]:
india_infrastructure = gpd.GeoDataFrame(
    pd.concat(
        all_regions,
        ignore_index=True
    ),
    crs="EPSG:4326"
)

print(india_infrastructure.shape)

(47561, 41)


In [9]:
columns_to_keep = [

    "amenity",

    "name",

    "operator",

    "healthcare",

    "emergency",

    "geometry",

    "RegionFile"

]

available_columns = [

    c

    for c in columns_to_keep

    if c in india_infrastructure.columns

]

india_infrastructure = (

    india_infrastructure[
        available_columns
    ]

)

In [10]:
india_infrastructure = (
    india_infrastructure[
        india_infrastructure.geometry.notnull()
    ]
)

india_infrastructure = (
    india_infrastructure[
        ~india_infrastructure.geometry.is_empty
    ]
)

print(
    "Remaining:",
    len(india_infrastructure)
)

Remaining: 47561


In [11]:
india_infrastructure = (
    india_infrastructure.drop_duplicates(
        subset=[
            "amenity",
            "name",
            "geometry"
        ]
    )
)

print(
    "After duplicates:",
    len(india_infrastructure)
)

After duplicates: 47537


In [12]:
print()

print(
    india_infrastructure["amenity"].value_counts()
)

print()

print(
    india_infrastructure.info()
)


amenity
hospital        26790
clinic          10508
school           9912
fire_station      327
Name: count, dtype: int64

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 47537 entries, 0 to 47560
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   amenity     47537 non-null  object  
 1   name        44846 non-null  object  
 2   operator    1708 non-null   object  
 3   geometry    47537 non-null  geometry
 4   RegionFile  47537 non-null  object  
dtypes: geometry(1), object(4)
memory usage: 2.2+ MB
None


In [13]:
india_infrastructure["geometry"] = (
    india_infrastructure.geometry.representative_point()
)

In [14]:
india_infrastructure.to_parquet(
    OUTPUT_FILE,
    index=False
)

print()

print("Saved to")

print(OUTPUT_FILE)


Saved to
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\india_infrastructure.parquet


In [15]:
loaded = gpd.read_parquet(
    OUTPUT_FILE
)

print(loaded.head())

print()

print(loaded.shape)

    amenity                                 name         operator  \
0    school                    Bijra High School             None   
1    school                       Chilika School             None   
2    school           Adarsh Gyanodaya Vidyalaya  Adarsh Gyanoday   
3  hospital                        Eden Hospital             None   
4  hospital  Calcutta Medical College & Hospital             None   

                    geometry                   RegionFile  
0  POINT (87.34798 23.58169)  eastern-zone-260710.osm.pbf  
1   POINT (85.1866 19.71314)  eastern-zone-260710.osm.pbf  
2   POINT (85.70399 25.4072)  eastern-zone-260710.osm.pbf  
3  POINT (88.36002 22.57274)  eastern-zone-260710.osm.pbf  
4  POINT (88.36144 22.57222)  eastern-zone-260710.osm.pbf  

(47537, 5)


In [16]:
print(india_infrastructure["amenity"].value_counts())

amenity
hospital        26790
clinic          10508
school           9912
fire_station      327
Name: count, dtype: int64
